# HSV Preprocessing Ablation

**Status: Negative result — not included in the final paper.**

Tests whether HSV colour-space preprocessing improves accuracy over standard RGB
for the four models in the paper (InceptionV3, DenseNet121, Xception, MobileNetV2).

**Conclusion:** HSV did not improve accuracy for any model. RGB normalisation
was used in the final paper.

Related paper: Joy et al., ICoEIT 2025, IEEE. DOI: 10.1109/ICoEIT.2025.01

## 0. Configuration — set your path here

In [ ]:
import os

# ── Edit this line only ───────────────────────────────────────────────────────
DATA_DIR = "path/to/newplantdisease_subset"
# ─────────────────────────────────────────────────────────────────────────────

TRAIN_DIR   = os.path.join(DATA_DIR, "train")
VALID_DIR   = os.path.join(DATA_DIR, "valid")
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 10

print("Config loaded.")

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import InceptionV3, DenseNet121, Xception, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow: {tf.__version__}")

## 2. HSV preprocessing function

In [ ]:
def rgb_to_hsv_preprocess(img: np.ndarray) -> np.ndarray:
    """
    Convert RGB image to HSV and normalise to [0, 1].
    Used as preprocessing_function in ImageDataGenerator.

    Args:
        img: np.ndarray (H, W, 3), values in [0, 255].
    Returns:
        HSV image normalised to [0, 1], shape (H, W, 3).
    """
    img_uint8 = img.astype(np.uint8)
    hsv = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 0] /= 179.0   # OpenCV Hue range is [0, 179]
    hsv[:, :, 1] /= 255.0
    hsv[:, :, 2] /= 255.0
    return hsv

print("HSV preprocessing function defined.")

## 3. Data generators

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=rgb_to_hsv_preprocess,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)
valid_datagen = ImageDataGenerator(preprocessing_function=rgb_to_hsv_preprocess)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True,
)
valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)

num_classes = len(train_generator.class_indices)
print(f"Classes: {num_classes}")

## 4. Model builder

In [ ]:
def create_model(base_model, dropout: float = 0.5):
    """Frozen-base transfer learning model with custom head."""
    base_model.trainable = False
    x   = GlobalAveragePooling2D()(base_model.output)
    x   = Dense(512, activation="relu")(x)
    x   = Dropout(dropout)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=out)

print("Model builder defined.")

## 5. Train InceptionV3 with HSV

In [ ]:
inception_base = InceptionV3(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
inception_model = create_model(inception_base, dropout=0.6)
inception_model.compile(optimizer=Adam(0.0005), loss="categorical_crossentropy", metrics=["accuracy"])

checkpoint    = ModelCheckpoint("InceptionV3_hsv.keras", monitor="val_accuracy", save_best_only=True, mode="max")
early_stop    = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
reduce_lr     = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)

print("Training InceptionV3 (Phase 1 — frozen base)...")
inception_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=30, callbacks=[checkpoint, early_stop, reduce_lr],
)

print("Fine-tuning InceptionV3 (Phase 2 — last 50 layers)...")
for layer in inception_base.layers[-50:]:
    layer.trainable = True
inception_model.compile(optimizer=Adam(0.0001), loss="categorical_crossentropy", metrics=["accuracy"])
inception_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=15, callbacks=[checkpoint, early_stop],
)

## 6. Train DenseNet121 with HSV

In [ ]:
densenet_base  = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
densenet_model = create_model(densenet_base, dropout=0.5)
densenet_model.compile(optimizer=Adam(0.0005), loss="categorical_crossentropy", metrics=["accuracy"])

checkpoint = ModelCheckpoint("DenseNet121_hsv.keras", monitor="val_accuracy", save_best_only=True, mode="max")
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)

print("Training DenseNet121 (Phase 1 — frozen base)...")
densenet_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=30, callbacks=[checkpoint, early_stop, reduce_lr],
)

print("Fine-tuning DenseNet121 (Phase 2 — last 12 layers)...")
for layer in densenet_base.layers[-12:]:
    layer.trainable = True
densenet_model.compile(optimizer=Adam(0.0002), loss="categorical_crossentropy", metrics=["accuracy"])
densenet_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=10, callbacks=[checkpoint, early_stop],
)

## 7. Train Xception with HSV

In [ ]:
xception_base  = Xception(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
xception_model = create_model(xception_base, dropout=0.5)
xception_model.compile(optimizer=Adam(0.0005), loss="categorical_crossentropy", metrics=["accuracy"])

checkpoint = ModelCheckpoint("Xception_hsv.keras", monitor="val_accuracy", save_best_only=True, mode="max")
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)

print("Training Xception (Phase 1 — frozen base)...")
xception_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=30, callbacks=[checkpoint, early_stop, reduce_lr],
)

print("Fine-tuning Xception (Phase 2 — last 15 layers)...")
for layer in xception_base.layers[-15:]:
    layer.trainable = True
xception_model.compile(optimizer=Adam(0.0002), loss="categorical_crossentropy", metrics=["accuracy"])
xception_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=10, callbacks=[checkpoint, early_stop],
)

## 8. Train MobileNetV2 with HSV

In [ ]:
mobilenet_base  = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
mobilenet_model = create_model(mobilenet_base, dropout=0.5)
mobilenet_model.compile(optimizer=Adam(0.0005), loss="categorical_crossentropy", metrics=["accuracy"])

checkpoint = ModelCheckpoint("MobileNetV2_hsv.keras", monitor="val_accuracy", save_best_only=True, mode="max")
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)

print("Training MobileNetV2 (Phase 1 — frozen base)...")
mobilenet_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=30, callbacks=[checkpoint, early_stop, reduce_lr],
)

print("Fine-tuning MobileNetV2 (Phase 2 — all layers unfrozen)...")
mobilenet_base.trainable = True
mobilenet_model.compile(optimizer=Adam(0.0001), loss="categorical_crossentropy", metrics=["accuracy"])
mobilenet_model.fit(
    train_generator, validation_data=valid_generator,
    epochs=10, callbacks=[checkpoint, early_stop],
)

## 9. Results — RGB (paper) vs HSV

In [ ]:
# Fill in your HSV val_accuracy results after running sections 5–8
RGB_RESULTS = {
    "InceptionV3":  0.9747,
    "DenseNet121":  0.9806,
    "Xception":     0.9760,
    "MobileNetV2":  0.9847,
}
HSV_RESULTS = {
    "InceptionV3":  None,   # fill after training
    "DenseNet121":  None,
    "Xception":     None,
    "MobileNetV2":  None,
}

print(f"{'Model':<16} {'RGB (paper)':>12} {'HSV':>8} {'Δ':>8}")
print("-" * 48)
for model_name in RGB_RESULTS:
    rgb = RGB_RESULTS[model_name]
    hsv = HSV_RESULTS[model_name]
    hsv_str   = f"{hsv:.4f}" if hsv else "pending"
    delta_str = f"{hsv - rgb:+.4f}" if hsv else "—"
    print(f"{model_name:<16} {rgb:.4f}       {hsv_str:>8} {delta_str:>8}")